<a href="https://colab.research.google.com/github/irAbs174/openshell-notebook/blob/main/openshell_notobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="باز کردن در Colab"/></a>

# جداسازی مدرن عامل‌های هوش مصنوعی با NVIDIA OpenShell: معماری کامل و دفترچه یادداشت عملی گام‌به‌گام

**NVIDIA OpenShell** یک محیط اجرای منبع‌باز و مبتنی بر سیاست است که به‌طور خاص برای عامل‌های هوش مصنوعی خودمختار و خودتکامل‌یابنده طراحی شده است. برخلاف محیط‌های اجرای کانتینر سنتی که بارهای کاری عمومی را ایزوله می‌کنند، OpenShell **محدودیت‌های محیطی خارج از فرایند و بدون اعتماد** را فراهم می‌کند.

یک عامل هوش مصنوعی که در داخل جعبه‌شنی OpenShell اجرا می‌شود، می‌تواند خود را تکامل دهد، اسکریپت‌های پایتون جدیدی در حین کار بنویسد و ابزارها را نصب کند—اما در عین حال کاملاً ناتوان از استخراج اعتبارنامه‌ها، عبور از مسیرهای غیرمجاز سیستم فایل یا برقراری درخواست‌های شبکه خروجی تأییدنشده است.

---

## ۱. معماری عمیق و مفاهیم اصلی

OpenShell از مدل امنیتی مشابه با جعبه‌شنی زبانه‌های مرورگر وب مدرن استفاده می‌کند:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           محیط میزبان                                        │
│                                                                             │
│  ┌───────────────────────┐             ┌─────────────────────────────────┐  │
│  │     CLI OpenShell     │             │        openshell-gateway        │  │
│  │  (`openshell sandbox`)│             │   (سرور gRPC / HTTP :۱۷۶۷۰)    │  │
│  └───────────┬───────────┘             └────────────────┬────────────────┘  │
│              │                                          │                   │
│              └──────────────────┐  ┌────────────────────┘                   │
│                                 ▼  ▼                                        │
│  ┌───────────────────────────────────────────────────────────────────────┐  │
│  │                    محیط اجرای جعبه‌شنی                                │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐  │  │
│  │  │                        چارچوب عامل                             │  │  │
│  │  │                 (Claude Code, Codex, سفارشی)                   │  │  │
│  │  └──────────────────────────────┬──────────────────────────────────┘  │  │
│  │                                 │ (اجرای دستورات)                     │  │
│  │                                 ▼                                     │  │
│  │  ┌─────────────────────────────────────────────────────────────────┐  │  │
│  │  │                    سیاست خارج از فرایند                         │  │  │
│  │  ├─────────────────────────────────────────────────────────────────┤  │  │
│  │  │ ۱. لایه سیستم فایل : جعبه‌شنی Landlock لینوکس                  │  │  │
│  │  │ ۲. لایه شبکه       : فیلتر پروکسی L7 (میزبان، API، خروجی)     │  │  │
│  │  │ ۳. مسیریاب حریم خصوصی : تعویض اعتبارنامه LLM بدون اعتماد      │  │  │
│  │  └─────────────────────────────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────────────────────────────┘  │  │
└─────────────────────────────────────────────────────────────────────────────┘

```

### ستون‌های معماری کلیدی

۱. **اعمال خارج از فرایند**: کنترل‌های امنیتی *خارج* از پنجره زمینه و مرز فرایند عامل زندگی می‌کنند. حتی اگر مهاجم تزریق پرامپت موفقی علیه عامل انجام دهد، هسته و دروازه OpenShell از فراخوانی‌های سیستمی ممنوع و مسیرهای شبکه غیرمجاز خودداری می‌کنند.
۲. **ایزوله‌سازی سیستم فایل Landlock لینوکس**: قوانین سطح دایرکتوری خواندن/نوشتن را در سطح هسته لینوکس اعمال می‌کند.
۳. **پروکسی L7 و فیلتر شبکه**: درخواست‌های HTTP/HTTPS خروجی از یک پروکسی شفاف مبتنی بر سیاست عبور می‌کنند. نقاط پایانی تأییدنشده بلافاصله با کد وضعیت `۴۰۳` مسدود می‌شوند.
۴. **مسیریاب حریم خصوصی (ایزوله‌سازی اعتبارنامه)**: کلیدهای API ارائه‌دهنده LLM را کاملاً خارج از جعبه‌شنی نگه می‌دارد. عامل درخواست‌های استنتاج محلی را به مسیر داخلی هدایت می‌کند و OpenShell اعتبارنامه‌های پشتیبان مجاز را در حین پرواز جایگزین می‌کند.

---

## ۲. دفترچه یادداشت Jupyter تعاملی با کارایی بالا

سلول‌های کد زیر را مستقیماً در یک فایل `.ipynb` کپی و جای‌گذاری کنید. اجرای متوالی سلول‌ها شما را از صفر تا اجرای یک عامل ایزوله با سیاست‌های اعلامی سفارشی راهنمایی می‌کند.

---

### سلول Markdown ۱: راه‌اندازی محیط و عیب‌یابی

```markdown
# بخش ۱: عیب‌یابی سیستم OpenShell و تأیید محیط
در این بخش، محیط میزبان را بررسی می‌کنیم، درایورهای محاسباتی موجود (Docker، Podman یا Kubernetes) را بررسی کرده و فایل‌های باینری OpenShell را بازرسی می‌کنیم.

```

### سلول کد ۱

In [ ]:
import os
import shutil
import subprocess
import sys


def run_command(cmd, verbose=True):
  """دستورات پوسته را اجرا کرده و خروجی را به‌طور امن در Jupyter پخش می‌کند."""
  print(f"\033[1;34m[اجرا]\033[0m {cmd}")
  process = subprocess.Popen(
      cmd,
      shell=True,
      stdout=subprocess.PIPE,
      stderr=subprocess.PIPE,
      text=True,
  )

  stdout_lines, stderr_lines = [], []
  while True:
    output = process.stdout.readline()
    if output == "" and process.poll() is not None:
      break
    if output and verbose:
      print(output.strip())
      stdout_lines.append(output)

  _, stderr = process.communicate()
  if stderr and verbose:
    print(f"\033[1;31m[خطا]\033[0m\n{stderr.strip()}")

  return process.returncode, "".join(stdout_lines), stderr


# ۱. بررسی ابزارهای سیستم
tools = ["openshell", "openshell-gateway", "docker", "curl"]
for tool in tools:
  path = shutil.which(tool)
  status = f"\033[1;32mیافت شد\033[0m در {path}" if path else "\033[1;31mیافت نشد\033[0m"
  print(f"بررسی ابزار [{tool}]: {status}")

# ۲. بررسی نسخه‌های OpenShell
run_command("openshell --version")
run_command("openshell-gateway --version")

---

### سلول Markdown ۲: راه‌اندازی و پیکربندی `openshell-gateway`

```markdown
# بخش ۲: راه‌اندازی سرویس دروازه OpenShell
`openshell-gateway` یک دیمن پس‌زمینه gRPC/HTTP است که چرخه‌های حیات جعبه‌شنی، گواهی‌های مشتری mTLS و ارزیابی سیاست‌ها را مدیریت می‌کند.

```

### سلول کد ۲

In [ ]:
import time

# تعریف پیکربندی محیط برای دروازه توسعه محلی
gateway_port = 17670
gateway_env = os.environ.copy()
gateway_env["OPENSHELL_BIND_ADDRESS"] = "127.0.0.1"
gateway_env["OPENSHELL_SERVER_PORT"] = str(gateway_port)
gateway_env["OPENSHELL_LOG_LEVEL"] = "info"
gateway_env["OPENSHELL_DISABLE_TLS"] = "true"  # حالت توسعه محلی

print("در حال راه‌اندازی openshell-gateway در پس‌زمینه...")
gateway_proc = subprocess.Popen(
    [
        "openshell-gateway",
        "--bind-address",
        "127.0.0.1",
        "--port",
        str(gateway_port),
        "--disable-tls",
        "--log-level",
        "info",
    ],
    env=gateway_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

# منتظر مقداردهی اولیه دروازه
time.sleep(3)

# تست اتصال به دروازه با استفاده از CLI doctor/status
code, out, err = run_command(
    f"openshell --gateway-endpoint http://127.0.0.1:{gateway_port} gateway"
    " doctor"
)

# ثبت دروازه محلی در پیکربندی فراداده
run_command(
    f"openshell gateway add http://127.0.0.1:{gateway_port} --local"
    " --name local-dev"
)

---

### سلول Markdown ۳: ایجاد و بازرسی جعبه‌شنی

```markdown
# بخش ۳: مدیریت چرخه حیات جعبه‌شنی
ما یک محیط اجرای ایزوله ایجاد کرده و محدودیت‌های امنیتی پیش‌فرض را بررسی می‌کنیم.

```

### سلول کد ۳

In [ ]:
# ۱. ایجاد یک جعبه‌شنی ایزوله با نام 'agent-demo'
run_command("openshell sandbox create --name agent-demo")

# ۲. فهرست جعبه‌شنی‌های فعال
run_command("openshell sandbox list")

# ۳. تست دسترسی به شبکه خروجی در داخل جعبه‌شنی پیش‌فرض
# اتصالات خروجی باید توسط موتور سیاست L7 به‌طور پیش‌فرض رد شوند.
test_net_cmd = (
    "openshell exec agent-demo -- curl -sS --connect-timeout 5"
    " https://api.github.com/zen"
)
run_command(test_net_cmd)

---

### سلول Markdown ۴: نوشتن و اعمال سیاست‌های اعلامی

```markdown
# بخش ۴: تزریق پویای سیاست
سیاست‌های OpenShell بر سیستم فایل، مسیرهای شبکه L7 و اجرای فرایندها بدون نیاز به راه‌اندازی مجدد جعبه‌شنی حاکم هستند.

```

### سلول کد ۴

In [ ]:
# ایجاد یک فایل YAML سیاست سختگیرانه
policy_content = """
apiVersion: v1alpha1
kind: SandboxPolicy
metadata:
  name: demo-github-read-only
spec:
  network:
    egress:
      - match:
          host: "api.github.com"
          scheme: "https"
        rules:
          - methods: ["GET"]
            path: "/*"
            action: Allow
          - methods: ["POST", "PUT", "DELETE"]
            path: "/*"
            action: Deny
  filesystem:
    readOnlyPaths:
      - "/usr"
      - "/lib"
    readWritePaths:
      - "/tmp"
      - "/workspace"
  process:
    allowExecution:
      - "/bin/*"
      - "/usr/bin/*"
"""

policy_filename = "github_policy.yaml"
with open(policy_filename, "w") as f:
  f.write(policy_content.strip())

print(f"فایل سیاست ایجاد شد: {policy_filename}")

# اعمال پویای سیاست به جعبه‌شنی در حال اجرا
run_command(f"openshell policy set agent-demo --policy {policy_filename} --wait")

---

### سلول Markdown ۵: تأیید کنترل‌های امنیتی (تست‌های خروجی و متد)

```markdown
# بخش ۵: تأیید اعمال سیاست
ما HTTP GET (مجاز) را در مقابل HTTP POST (ممنوع) در `api.github.com` تست می‌کنیم.

```

### سلول کد ۵

In [ ]:
print("--- تست ۱: HTTP GET (باید موفق باشد) ---")
run_command(
    "openshell exec agent-demo -- curl -sS -X GET https://api.github.com/zen"
)

print("\n--- تست ۲: HTTP POST (باید توسط پروکسی L7 مسدود شود) ---")
run_command(
    "openshell exec agent-demo -- curl -sS -X POST"
    " https://api.github.com/user/repos -d '{\"name\":\"exploit\"}'"
)

print("\n--- تست ۳: میزبان خروجی غیرمجاز (باید مسدود شود) ---")
run_command(
    "openshell exec agent-demo -- curl -sS --connect-timeout 3"
    " https://example.com"
)

---

### سلول Markdown ۶: مشاهده لاگ‌های حسابرسی جعبه‌شنی و پاک‌سازی

```markdown
# بخش ۶: لاگ‌گیری حسابرسی امنیتی و پاک‌سازی جعبه‌شنی
OpenShell هر درخواست رهگیری‌شده، نقض سیاست و رویداد جعبه‌شنی را ثبت می‌کند.

```

### سلول کد ۶

In [ ]:
# دریافت لاگ‌های امنیتی
print("=== لاگ‌های حسابرسی جعبه‌شنی ===")
run_command("openshell logs agent-demo --tail 20")

# پاک‌سازی منابع جعبه‌شنی
print("\n=== پاک‌سازی ===")
run_command("openshell sandbox delete agent-demo --force")

# متوقف کردن فرایند دروازه
if 'gateway_proc' in locals():
  gateway_proc.terminate()
  print("سرور OpenShell Gateway با موفقیت متوقف شد.")

---

## ۳. مرجع سریع دستورات CLI و دروازه

| زیردستور OpenShell | توضیحات |
| --- | --- |
| `openshell sandbox create` | ایجاد یک جعبه‌شنی اجرای ایزوله جدید |
| `openshell policy set <نام> -p <فایل>` | اعمال سیاست‌های YAML با بارگذاری مجدد داغ به جعبه‌شنی |
| `openshell logs <نام>` | نمایش لاگ‌های حسابرسی شبکه و سیستم فایل به‌صورت بی‌درنگ |
| `openshell-gateway --config <فایل>` | راه‌اندازی دیمن دروازه اجرای مرکزی |
| `openshell-gateway generate-certs` | تولید گواهی‌های PKI برای احراز هویت mTLS |

---

## ۴. الگوهای پیشرفته تولید و قابلیت‌های سازمانی

برای تکمیل یکپارچه‌سازی OpenShell مبتنی بر Jupyter، در اینجا سه الگوی ضروری برای اجرای بارهای کاری عامل در محیط تولید آورده شده است.

---

### الگوی الف: احراز هویت OIDC و JWT چندمستاجری

هنگام استقرار `openshell-gateway` در یک خوشه چندکاربره یا پلتفرم سازمانی، احراز هویت از گواهی‌های محلی mTLS به اعتبارسنجی JWT از طریق OpenID Connect (OIDC) تغییر می‌کند.

#### پیکربندی راه‌اندازی دروازه

```bash
openshell-gateway \
  --bind-address 0.0.0.0 \
  --port 17670 \
  --oidc-issuer https://auth.yourcompany.com/realms/agents \
  --oidc-audience openshell-cli \
  --oidc-roles-claim realm_access.roles \
  --oidc-admin-role openshell-admin \
  --oidc-user-role openshell-user \
  --drivers kubernetes

```

#### اجرای دستورات احراز هویت‌شده در پایتون

In [ ]:
import os
import subprocess

# دریافت توکن دسترسی JWT از ارائه‌دهنده هویت (مثلاً Keycloak، Entra ID، Okta)
jwt_token = os.environ.get("OPENID_ACCESS_TOKEN")

# نقطه پایانی دروازه هدف با ارسال توکن حامل
cmd = [
    "openshell",
    "--gateway-endpoint",
    "https://openshell-gateway.internal.net:17670",
    "sandbox",
    "create",
    "--name",
    "authenticated-user-agent",
]

env = os.environ.copy()
env["OPENSHELL_BEARER_TOKEN"] = jwt_token

result = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(result.stdout)

---

### الگوی ب: لوله‌سازی چندمرحله‌ای ایزوله عامل (اجرا و تأیید کد)

یک جریان کاری رایج در عامل‌های خودتکامل‌یابنده شامل جداسازی **تولید کد** از **اجرای کد غیرقابل اعتماد** است. شما می‌توانید دو جعبه‌شنی با سیاست‌های متفاوت در داخل Jupyter تنظیم کنید:

```
┌───────────────────────────┐         ┌───────────────────────────┐
│   جعبه‌شنی تولیدکننده    │         │   جعبه‌شنی اجراکننده      │
│  • شبکه: خروجی به LLM   │  ────►  │  • شبکه: ایزوله           │
│  • سیستم فایل: فقط نوشتن│  کد    │  • سیستم فایل: فقط خواندن │
└───────────────────────────┘         └───────────────────────────┘

```

In [ ]:
import subprocess


class DualStageAgentWorkflow:

  def __init__(self, gateway_url="http://127.0.0.1:17670"):
    self.gateway = gateway_url

  def run_pipeline(self, python_code_to_verify: str):
    # ۱. ایجاد جعبه‌شنی اجراکننده غیرقابل اعتماد
    subprocess.run(
        f"openshell --gateway-endpoint {self.gateway} sandbox create --name"
        " evaluator",
        shell=True,
        check=True,
    )

    # ۲. نوشتن سیاست آفلاین محدودکننده (بدون خروجی شبکه)
    offline_policy = """
apiVersion: v1alpha1
kind: SandboxPolicy
metadata:
  name: total-isolation
spec:
  network:
    egress: []  # اتصالات خروجی ممنوع
  filesystem:
    readOnlyPaths: ["/usr", "/lib"]
    readWritePaths: ["/tmp"]
"""
    with open("/tmp/offline_policy.yaml", "w") as f:
      f.write(offline_policy)

    subprocess.run(
        f"openshell --gateway-endpoint {self.gateway} policy set evaluator"
        " --policy /tmp/offline_policy.yaml --wait",
        shell=True,
        check=True,
    )

    # ۳. اجرای امن کد غیرقابل اعتماد در ارزیاب با اعتماد صفر
    exec_cmd = (
        f"openshell --gateway-endpoint {self.gateway} exec evaluator -- python3"
        f" -c '{python_code_to_verify}'"
    )
    res = subprocess.run(exec_cmd, shell=True, capture_output=True, text=True)

    print("=== خروجی مرحله ارزیابی ===")
    print("خروجی استاندارد:", res.stdout)
    print("خطای استاندارد:", res.stderr)
    print("کد بازگشت:", res.returncode)

    # ۴. پاک‌سازی
    subprocess.run(
        f"openshell --gateway-endpoint {self.gateway} sandbox delete evaluator"
        " --force",
        shell=True,
    )


# اجرای تست لوله
pipeline = DualStageAgentWorkflow()
pipeline.run_pipeline("import sys; print('ارزیابی امن کد غیرقابل اعتماد!')")

---

### الگوی ج: یکپارچه‌سازی مشاهده‌پذیری OpenShell با معیارهای Prometheus

هنگامی که `openshell-gateway` با پارامتر `--metrics-port 9090` راه‌اندازی می‌شود، یک نقطه پایانی معیارهای Prometheus را برای نظارت بر جعبه‌شنی‌ها، نقض‌های خروجی و عملکرد سیستم به‌صورت بی‌درنگ ارائه می‌دهد.

In [ ]:
import urllib.request

# درخواست به نقطه پایانی معیارهای Prometheus OpenShell
metrics_url = "http://127.0.0.1:9090/metrics"

try:
  with urllib.request.urlopen(metrics_url) as response:
    metrics_data = response.read().decode("utf-8")

  # فیلتر معیارها برای نقض‌های خروجی و جعبه‌شنی‌های فعال
  relevant_metrics = [
      line
      for line in metrics_data.split("\n")
      if "openshell_sandbox" in line or "openshell_policy_violations" in line
  ]

  print("=== معیارهای اجرای OpenShell ===")
  for metric in relevant_metrics[:10]:
    print(metric)

except Exception as e:
  print(f"نقطه پایانی معیارها غیرفعال یا غیرقابل دسترس است: {e}")